# Outlier Analysis

Identifies morphological outliers using Mahalanobis distance in PC1–PC2 space.
Threshold: mean + 3 SD of Mahalanobis distances across all retained masks.


## Setup

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from scipy.spatial.distance import mahalanobis

DATA_DIR = Path('../data')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

outliers = pd.read_csv(DATA_DIR / 'outliers.csv')
print(f'Total samples: {len(outliers)}')
print(f'Outliers (is_outlier=True): {outliers["is_outlier"].sum()}')


## Mahalanobis distance distribution

In [ ]:
threshold = outliers['mahalanobis_dist'].mean() + 3 * outliers['mahalanobis_dist'].std()

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(outliers['mahalanobis_dist'], bins=50, color='steelblue', edgecolor='white', linewidth=0.3)
ax.axvline(threshold, color='crimson', linestyle='--', label=f'Threshold (mean+3SD) = {threshold:.2f}')
ax.set_xlabel('Mahalanobis distance')
ax.set_ylabel('Count')
ax.set_title('Mahalanobis distance distribution in PC1–PC2 space')
ax.legend()
plt.tight_layout(); plt.show()
print(f'Threshold: {threshold:.3f}')
print(f'Outliers: {(outliers["mahalanobis_dist"] > threshold).sum()}')


## Outlier scatter in PC1–PC2 space

In [ ]:
clean = outliers[~outliers['is_outlier']]
out   = outliers[outliers['is_outlier']]

fig, ax = plt.subplots(figsize=(8, 7))
ax.scatter(clean['pc1'], clean['pc2'], alpha=0.4, s=10, color='steelblue', label='Clean', linewidths=0)
ax.scatter(out['pc1'],   out['pc2'],   alpha=0.9, s=35, color='crimson',   label='Outlier', zorder=5)
for _, row in out.iterrows():
    label = row['image_id'].replace('_mask', '').replace('.png', '')
    ax.annotate(label, (row['pc1'], row['pc2']), fontsize=6.5, ha='left',
                xytext=(4, 2), textcoords='offset points')
ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
ax.set_title(f'Outlier detection: {len(out)} outliers from {len(outliers)} masks')
ax.legend()
plt.tight_layout(); plt.show()


## Top outliers

In [ ]:
top = out.sort_values('mahalanobis_dist', ascending=False).head(10)
top[['image_id','individual','region','pc1','pc2','mahalanobis_dist']].round(3)
